# [Azure OpenAI Responses API's](https://learn.microsoft.com/en-us/azure/ai-services/openai/how-to/responses?tabs=python-secure#generate-a-text-response) (with Entra ID authentication)

In [1]:
import os, sys
from openai import AzureOpenAI
from dotenv import load_dotenv # requires python-dotenv
from azure.identity import DefaultAzureCredential, AzureCliCredential, get_bearer_token_provider

# load ".env" from the kernel working folder, e.g. same folder where we ran "Jupyter notebook"
if not load_dotenv():
    print("Environment variables not loaded, cell execution stopped")
    sys.exit()

openai_api_version    = os.environ["AZURE_OPENAI_API_VERSION"]
azure_openai_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"]
azure_deployment_name = os.environ["AZURE_OPENAI_CHAT_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential(
    exclude_environment_credential=True
)


LANGUAGE = "English" # "English", "Italian", "French", "Japanese"...

token_provider = get_bearer_token_provider(
    credential,
    "https://cognitiveservices.azure.com/.default",
)

client = AzureOpenAI(
    azure_ad_token_provider=token_provider,
    api_version=openai_api_version,
    azure_endpoint=azure_openai_endpoint
)

print(f"openai_endpoint: {azure_openai_endpoint}")
print(f"azure_deployment_name: {azure_deployment_name}")
print(f"openai_api_version: {openai_api_version}")

openai_endpoint: https://mm-ai-upskilling-project-resourc.openai.azure.com/
azure_deployment_name: gpt-5.4-mini
openai_api_version: 2025-04-01-preview


# Read the system message plus the json list of metrics

In [2]:
import json

# Open the system_message file in read mode
# The file lives in the "notebook working folder" where the physical file .ipynb is stored
with open("system_message.txt", "r", encoding="utf-8") as file:
    system_message = file.read()

# Open the metrics file in read mode
with open("metrics.json", "r", encoding="utf-8") as file:
    metrics_list = json.loads(file.read().replace("\n", ""))

i=1
for m in metrics_list:
    print (f'{i}: {m["metric_name"]}')
    i += 1

1: Intent Resolution
2: Tool Call Accuracy
3: Task Adherence
4: Response Completeness
5: Groundedness (prompt-based)
6: Groundedness Pro
7: Retrieval
8: Relevance
9: Coherence
10: Fluency
11: Similarity
12: F1 Score
13: BLEU Score
14: ROUGE Score
15: METEOR Score


# You may run the next cell multiple times to explore metrics samples

In [3]:
from IPython.display import Markdown, display

index = int(input("Which metric would you like to analyze?"))

display(Markdown(f"\n\n**GREAT CHOICE!** Here are a couple examples for the metric `<{metrics_list[index-1]['metric_name']}>`\n"))

messages=[{"role": "system", "content": f"{system_message}.\nQUESTION, ANSWER, SCORE AND  EXPLANATION must be in {LANGUAGE}"},
          {"role": "user", "content": json.dumps(metrics_list[index-1])}]

response = client.responses.create(
    model=os.environ['AZURE_OPENAI_CHAT_DEPLOYMENT_NAME'],
    input=messages)

display(Markdown(response.output_text.replace("\n", "  \n")))

Which metric would you like to analyze? 10




**GREAT CHOICE!** Here are a couple examples for the metric `<Fluency>`


Below are **5 fluency examples** from **5 (best)** to **1 (worst)**.    
They are tailored to **Fluency**: grammar, readability, coherence, sentence smoothness, and clarity of writing.  
  
---  
  
## 5 — Excellent Fluency  
**QUESTION:** Rewrite this meeting note into a clear business summary: "Talked about sales. Many issues. Next steps discussed."    
**ANSWER:** The team reviewed sales performance, identified several key issues, and discussed next steps to address them.    
**SCORE:** 5    
**EXPLANATION:** The response is grammatically correct, concise, coherent, and easy to read. Ideas flow smoothly with strong clarity.  
  
---  
  
## 4 — Good Fluency  
**QUESTION:** Rewrite this email request professionally: "Send me the report soon because I need it."    
**ANSWER:** Please send me the report as soon as possible, since I need it for my review.    
**SCORE:** 4    
**EXPLANATION:** The writing is clear and natural, with only minor simplicity in vocabulary and structure. It reads smoothly overall.  
  
---  
  
## 3 — Moderate Fluency  
**QUESTION:** Rewrite this sentence for a project update: "We finish work and now we are checking the things."    
**ANSWER:** We finished the work, and now we are checking the things to make sure everything is okay.    
**SCORE:** 3    
**EXPLANATION:** The meaning is understandable, but the wording is somewhat awkward and repetitive. Grammar and style are acceptable, but not polished.  
  
---  
  
## 2 — Low Fluency  
**QUESTION:** Rewrite this marketing line: "Our product good and help many people save time."    
**ANSWER:** Our product is good and it help many peoples save time very much.    
**SCORE:** 2    
**EXPLANATION:** The sentence has grammar errors and unnatural word choice. The message is only partly clear and does not sound smooth.  
  
---  
  
## 1 — Very Poor Fluency  
**QUESTION:** Rewrite this business update: "Budget review completed, concerns noted, team follow."    
**ANSWER:** Budget done. Concern note. Team the follow is.    
**SCORE:** 1    
**EXPLANATION:** The response is hard to understand due to broken grammar, missing structure, and very poor readability. It is not fluent.  
  
---  
  
If you want, I can do the same format for the next metric in your list.